In [1]:
# ===============================
# 1 Imports
# ===============================
import spacy
from spacy.tokens import Doc, DocBin
from sklearn.model_selection import train_test_split
print(spacy.__version__)
import pandas as pd
import re
from tqdm import tqdm

3.8.11


In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
os.chdir("/content/drive/MyDrive/OICE-POS")

In [4]:
# ===============================
# 2  DataFrame
# ===============================
df = pd.read_parquet("./dl_xml_utd_corpus.parquet")

In [5]:
# ===============================
# 3 Create a unique line ID for grouping tokens into sequences
# ===============================
df["line_id"] = (
    df["file"].astype(str) + "_" +
    df["pb"].astype(str) + "_" +
    df["lb"].astype(str) + "_" +
    df["cb"].astype(str)
)

# ---------------------------
# 5 Group into sequences (sentences)
# ---------------------------
grouped = df.groupby("line_id").agg({
    "norm": lambda x: list(x),   # normalized tokens
    "lemma": lambda x: list(x),  # lemmas
    "utd": lambda x: list(x)     # POS tags
}).reset_index()

# Remove any rows with empty tokens, lemma, or POS
grouped = grouped.dropna(subset=["norm", "lemma", "utd"])
grouped = grouped[grouped["norm"].apply(lambda x: all(str(t).strip() != "" for t in x))]
grouped = grouped[grouped["lemma"].apply(lambda x: all(str(t).strip() != "" for t in x))]
grouped = grouped[grouped["utd"].apply(lambda x: all(str(t).strip() != "" for t in x))]

#------------------------------------
# 6 Split into train, eval, test, as 80-10-10
# First split: train vs temp
train_df, temp_df = train_test_split(grouped, test_size=0.2, random_state=42)

# Second split: dev vs test
dev_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(len(train_df), len(dev_df), len(test_df))

21653 2707 2707


In [6]:
grouped

,line_id,norm,lemma,utd
0,AM-132-fol-Bandamanna-saga.xml_114r_1.0_b,"[Ófeigr, hét, maðr, er, bjó, vestr]","[ófeigr, heita, maðr, er, búa, vestr]","[PROPN, VERB, NOUN, PART, VERB, ADV]"
1,AM-132-fol-Bandamanna-saga.xml_114r_10.0_b,"[mikill, ok, hinn, mesti, ráðagerðamaðr, ., Ha...","[mikill, ok, hinn, mikill, ráðagerðamaðr, None...","[ADJ, CCONJ, DET, ADJ, NOUN, PUNCT, PRON, VERB..."
2,AM-132-fol-Bandamanna-saga.xml_114r_11.0_b,"[,, en, ekki, var, honum, fjárhagrinn, hǿgr, ,...","[None, en, engi, vera, hann, fjárhagr, hǿgr, N...","[PUNCT, CCONJ, PRON, VERB, PRON, NOUN, ADJ, PU..."
3,AM-132-fol-Bandamanna-saga.xml_114r_12.0_b,"[lendur, miklar, en, minna, lausafé, ., Hann, ...","[lenda, mikill, en, lítill, lausafé, None, han...","[NOUN, ADJ, CCONJ, ADJ, NOUN, PUNCT, PRON, VERB]"
4,AM-132-fol-Bandamanna-saga.xml_114r_13.0_b,"[við, engan, mann, mat, ,, en, þó, var, mjǫk, ...","[við, engi, maðr, matr, None, en, þó, vera, mj...","[ADP, PRON, NOUN, NOUN, PUNCT, CCONJ, ADV, VER..."
...,...,...,...,...
36235,AM-677-4to.xml_9v_41.0_None,"[óumrǿðiligum, trum, .]","[óumrǿðiligr, tár, None]","[ADJ, NOUN, PUNCT]"
36236,AM-677-4to.xml_9v_5.0_None,"[brjóstum, þeirra, ., Ok, urðu, hjǫrtu, þeirra...","[brjóst, sá, None, ok, verða, hjarta, sá, glóa...","[NOUN, DET, PUNCT, CCONJ, VERB, NOUN, DET, VER..."
36238,AM-677-4to.xml_9v_7.0_None,"[er, sjálfr, st, sem, Johannes, mǽlti, :, Guð...","[vera, sjalfr, st, sem, johannes, mǽla, None,...","[VERB, ADJ, NOUN, SCONJ, PROPN, VERB, PUNCT, N..."
36239,AM-677-4to.xml_9v_8.0_None,"[hann, þegar, þann, er, hann, elskar, ., En, e...","[hann, þegar, sá, er, hann, elska, None, en, e...","[PRON, ADV, DET, PART, PRON, VERB, PUNCT, CCON..."


In [7]:
# ===============================
# 4 Convert grouped DataFrame to spaCy DocBin
# ===============================

def df_to_docbin(df, path):
    nlp = spacy.blank("xx")
    db = DocBin()

    for _, row in df.iterrows():
        tokens = row["norm"]
        doc = Doc(nlp.vocab, words=tokens)

        for i, token in enumerate(doc):
            token.tag_ = row["utd"][i]
            token.lemma_ = row["lemma"][i]

        db.add(doc)

    db.to_disk(path)

df_to_docbin(train_df, "train.spacy")
df_to_docbin(dev_df, "dev.spacy")
df_to_docbin(test_df, "test.spacy")

In [8]:
##### Double check just in case########
# Load  saved DocBin
db = DocBin().from_disk("./train.spacy")

# Load a blank pipeline for the same language
nlp = spacy.blank("xx")

# Get all docs from the DocBin
docs = list(db.get_docs(nlp.vocab))

print(f" Loaded {len(docs)} documents from DocBin\n")

# Inspect first few docs
for i, doc in enumerate(docs[:3]):
    print(f"--- Doc {i} ---")
    for token in doc:
        print(f"{token.text}\tPOS: {token.tag_}\tLEMMA: {token.lemma_}")

 Loaded 21653 documents from DocBin

--- Doc 0 ---
með	POS: ADP	LEMMA: með
langri	POS: ADJ	LEMMA: langr
venju	POS: NOUN	LEMMA: venja
,	POS: PUNCT	LEMMA: 
at	POS: CCONJ	LEMMA: at
þau	POS: DET	LEMMA: sá
hafna	POS: VERB	LEMMA: hafna
sinni	POS: DET	LEMMA: sinn
náttúru	POS: NOUN	LEMMA: náttúra
,	POS: PUNCT	LEMMA: 
ok	POS: CCONJ	LEMMA: ok
verða	POS: VERB	LEMMA: verða
hlýðin	POS: ADJ	LEMMA: hlýðinn
ok	POS: CCONJ	LEMMA: ok
mjúk	POS: ADJ	LEMMA: mjúkr
sínum	POS: DET	LEMMA: sinn
meistara	POS: NOUN	LEMMA: meistari
.	POS: PUNCT	LEMMA: 
Sigrat	POS: VERB	LEMMA: sigra
hafið	POS: VERB	LEMMA: hafa
--- Doc 1 ---
ok	POS: CCONJ	LEMMA: ok
gengr	POS: VERB	LEMMA: ganga
heim	POS: ADV	LEMMA: heim
at	POS: ADP	LEMMA: at
dyrum	POS: NOUN	LEMMA: dyrr
.	POS: PUNCT	LEMMA: 
Ok	POS: CCONJ	LEMMA: ok
heyrir	POS: VERB	LEMMA: heyra
hann	POS: PRON	LEMMA: hann
þat	POS: DET	LEMMA: sá
at	POS: CCONJ	LEMMA: at
margt	POS: ADJ	LEMMA: margr
er	POS: VERB	LEMMA: vera
manna	POS: NOUN	LEMMA: maðr
--- Doc 2 ---
þá	POS: ADV	LEMMA: þá
er	P

In [9]:
# ===============================
# 5 Create spaCy config
# ===============================
!python -m spacy init config config.cfg \
    --lang xx \
    --pipeline tagger,trainable_lemmatizer \
    --optimize accuracy \
    --force


⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: xx
- Pipeline: tagger, trainable_lemmatizer
- Optimize for: accuracy
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [10]:
with open("config.cfg", "r", encoding="utf8") as f:
    text = f.read()

text = re.sub(r"max_steps\s*=\s*\d+", "max_steps = 12000", text)
text = re.sub(r"patience\s*=\s*\d+", "patience = 1000", text)
text = re.sub(r"eval_frequency\s*=\s*\d+", "eval_frequency = 200", text)

with open("config.cfg", "w", encoding="utf8") as f:
    f.write(text)

In [11]:
# ===============================
# 9 Train the pipeline
# ===============================
!python -m spacy train config.cfg \
    --output ./output \
    --paths.train ./train.spacy \
    --paths.dev ./dev.spacy \
    --gpu-id 0  # remove if no GPU


ℹ Saving to output directory: output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'tagger', 'trainable_lemmatizer']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS TAGGER  LOSS TRAIN...  TAG_ACC  POS_ACC  LEMMA_ACC  SCORE 
---  ------  ------------  -----------  -------------  -------  -------  ---------  ------
  0       0          0.00        82.30          92.95    33.77     0.00      44.13    0.39
  0     200        671.14      6300.66        9130.90    86.53     0.00      77.62    0.82
  0     400       1074.93      4010.19        6349.57    91.51     0.00      84.62    0.88
  0     600       1301.41      3806.51        5959.58    92.97     0.00      88.42    0.91
  0     800       1531.58      3880.28        5790.06    94.35     0.00      90.30    0.92
  0    1000       1819.74      4067.21      

In [12]:
# ===============================
# 9 Quick test of the trained model
# ===============================
nlp2 = spacy.load("./output/model-best")

# Example test sentence: first line in grouped data
example_sentence = " ".join(grouped['norm'].iloc[0])
doc = nlp2(example_sentence)

for token in doc:
    print(f"Token: {token.text}, POS: {token.tag_}, Lemma: {token.lemma_}")

Token: Ófeigr, POS: PROPN, Lemma: ófeigr
Token: hét, POS: VERB, Lemma: heita
Token: maðr, POS: NOUN, Lemma: maðr
Token: er, POS: PART, Lemma: er
Token: bjó, POS: VERB, Lemma: búa
Token: vestr, POS: ADV, Lemma: vestr


In [13]:
#### SCORE
docbin = DocBin().from_disk("test.spacy")
docs = list(docbin.get_docs(nlp.vocab))


In [14]:
from spacy.training import Example

# 1 Load best model
nlp = spacy.load("output/model-best")

# 2 Load the gold test set
docbin = DocBin().from_disk("test.spacy")
gold_docs = list(docbin.get_docs(nlp.vocab))

# 3 Build Example objects
examples = []
for gold in tqdm(gold_docs):
    # Predict on the same text
    pred = nlp(gold.text)
    # Create an Example pairing the predicted doc with the gold doc
    example = Example(pred, gold)
    examples.append(example)

# 4 Evaluate
results = nlp.evaluate(examples)

100%|██████████| 2707/2707 [00:38<00:00, 70.43it/s]


In [15]:
# 5 Metrics
print(f"Tag accuracy (POS):  {results['tag_acc']:.4f}")
print(f"Lemma accuracy: {results['lemma_acc']:.4f}")
print("\nFull results:\n", results)

Tag accuracy (POS):  0.9603
Lemma accuracy: 0.9328

Full results:
 {'token_acc': None, 'token_p': None, 'token_r': None, 'token_f': None, 'tag_acc': 0.9603103551178752, 'lemma_acc': 0.9327907811725283, 'speed': 2839.9775505512657}
